In [3]:
# import cudf
# import cupy as cp

# print("cuDF version:", cudf.__version__)
# print("CuPy test:", cp.arange(5))

cuDF version: 25.10.00
CuPy test: [0 1 2 3 4]


In [1]:
import pandas as pd
import numpy as np

from numpy.lib.stride_tricks import sliding_window_view

from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [7]:
train_df = pd.read_parquet("../data/processed/casp9_features.parquet")
val_df = pd.read_parquet("../data/processed/validation_features.parquet")
test_df = pd.read_parquet("../data/processed/testing_features.parquet")

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (3337092, 51)
Val: (47602, 51)
Test: (24466, 51)


In [8]:
feature_cols = [c for c in train_df.columns if c.startswith("Evo") or c.startswith("AA_")]

print("Number of features:", len(feature_cols))

Number of features: 41


In [9]:
WINDOW = 5
WINDOW_SIZE = 2 * WINDOW + 1

print("Window size:", WINDOW_SIZE)

Window size: 11


In [10]:
def build_windows_fast(df):

    X_list = []
    y_list = []

    for pid, group in df.groupby("ProteinID"):

        group = group.sort_values("ResidueIndex")

        features = group[feature_cols].values.astype(np.float32)
        target = group["CentroidDist"].values.astype(np.float32)

        padded = np.pad(
            features,
            ((WINDOW, WINDOW), (0,0)),
            mode="constant"
        )

        windows = sliding_window_view(
            padded,
            (WINDOW_SIZE, features.shape[1])
        )

        windows = windows.reshape(len(group), -1)

        X_list.append(windows)
        y_list.append(target)

    X = np.vstack(X_list)
    y = np.concatenate(y_list)

    return X, y

In [11]:
X_train_w, y_train_w = build_windows_fast(train_df)
print("Train windows:", X_train_w.shape)

X_val_w, y_val_w = build_windows_fast(val_df)
print("Val windows:", X_val_w.shape)

X_test_w, y_test_w = build_windows_fast(test_df)
print("Test windows:", X_test_w.shape)

Train windows: (3337092, 451)
Val windows: (47602, 451)
Test windows: (24466, 451)


Saving the windows

In [12]:
np.save("X_train.npy", X_train_w)
np.save("y_train.npy", y_train_w)

np.save("X_val.npy", X_val_w)
np.save("y_val.npy", y_val_w)

np.save("X_test.npy", X_test_w)
np.save("y_test.npy", y_test_w)

In [13]:
X_train_np = X_train_w
y_train_np = y_train_w

X_val_np = X_val_w
y_val_np = y_val_w

X_test_np = X_test_w
y_test_np = y_test_w

GPU accelerated XGBoost

In [16]:
xgb_model = XGBRegressor(

    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    tree_method="hist",   # new API
    device="cuda",        # enables GPU

    max_bin=256,

    random_state=42,
    verbosity=1
)

In [17]:
print("Training XGBoost with sequence windows on GPU...")

xgb_model.fit(
    X_train_np,
    y_train_np,
    eval_set=[(X_val_np, y_val_np)],
    verbose=True
)

Training XGBoost with sequence windows on GPU...
[0]	validation_0-rmse:837.87204
[1]	validation_0-rmse:832.33854
[2]	validation_0-rmse:827.42674
[3]	validation_0-rmse:822.79905
[4]	validation_0-rmse:818.68965
[5]	validation_0-rmse:814.93755
[6]	validation_0-rmse:811.67542
[7]	validation_0-rmse:808.54165
[8]	validation_0-rmse:805.85548
[9]	validation_0-rmse:802.95556
[10]	validation_0-rmse:800.41562
[11]	validation_0-rmse:798.35579
[12]	validation_0-rmse:796.27269
[13]	validation_0-rmse:794.57442
[14]	validation_0-rmse:793.05124
[15]	validation_0-rmse:791.23786
[16]	validation_0-rmse:789.75773
[17]	validation_0-rmse:788.47620
[18]	validation_0-rmse:787.00288
[19]	validation_0-rmse:785.78327
[20]	validation_0-rmse:784.77604
[21]	validation_0-rmse:784.10566
[22]	validation_0-rmse:783.04080
[23]	validation_0-rmse:782.15249
[24]	validation_0-rmse:781.46310
[25]	validation_0-rmse:780.96840
[26]	validation_0-rmse:780.17720
[27]	validation_0-rmse:779.87545
[28]	validation_0-rmse:779.51508
[29]

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=256, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [18]:
preds = xgb_model.predict(X_test_np)

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [10:09:18] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [19]:
rmse = np.sqrt(mean_squared_error(y_test_np, preds))
mae = mean_absolute_error(y_test_np, preds)
r2 = r2_score(y_test_np, preds)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 855.1081802906577
MAE: 577.7850341796875
R2: 0.14737212657928467


In [20]:
import joblib

joblib.dump(xgb_model, "xgboost_window_model.pkl")

['xgboost_window_model.pkl']